# TD 5 — Corrigé

**Analyse des données — L3 Économie**

Mis en ligne après la séance.

In [ ]:
import pandas as pd
import numpy as np
from scipy import stats

URL = ("https://raw.githubusercontent.com/StefaniaMarcassa/"
       "analyse_des_donnees/main/data/cm03-salaires.csv")

def en_nombre(serie):
    return pd.to_numeric(serie.astype(str).str.replace(" ", "", regex=False)
                                          .str.replace(",", ".", regex=False), errors="coerce")

df = pd.read_csv(URL, sep=";", encoding="utf-8")
df["SALAIRE_NET"] = en_nombre(df["SALAIRE_NET"]).replace(999999, np.nan)
df = df[df["SALAIRE_NET"].isna() | (df["SALAIRE_NET"] > 0)].copy()
s  = df["SALAIRE_NET"].dropna()

print(len(df), "lignes |", len(s), "salaires observes")

## Partie 1 — Décrire la distribution

In [ ]:
s.quantile([.1, .2, .3, .4, .5, .6, .7, .8, .9]).round(0)

**Réponse 1.** $D_1$ = **1 201 €**, $D_5$ = **2 322 €**, $D_9$ = **4 252 €**.

Un salarié sur dix gagne moins de 1 201 euros, un sur dix plus de 4 252. La médiane, 2 322 euros, est nettement inférieure à la moyenne de 2 610 : la distribution est **asymétrique à droite**.

In [ ]:
d1, d9 = s.quantile([.1, .9])
print(round(d9 / d1, 2))

**Réponse 2.** **3,54**.

Les 10 % les mieux payés gagnent au moins 3,5 fois ce que gagnent au plus les 10 % les moins payés.

Attention à la formulation : le rapport interdécile compare deux **seuils**, non deux moyennes. Il ne dit rien de ce qui se passe au-dessus de $D_9$ — ce qui, sur une distribution de salaires, est précisément là où l'inégalité se joue.

In [ ]:
print(round(100 * s[s >= d9].sum() / s.sum(), 1), "% pour les 10 % les mieux payes")
print(round(100 * s[s <= d1].sum() / s.sum(), 1), "% pour les 10 % les moins payes")

**Réponse 3.** **22,4 %** de la masse salariale pour les 10 % du haut, **3,6 %** pour les 10 % du bas — un rapport de six.

Ce chiffre dit quelque chose que le rapport interdécile ne dit pas : il porte sur la **masse**, donc sur ce qui se passe à l'intérieur de chaque dixième.

## Partie 2 — L'indice de Gini

In [ ]:
def gini(x):
    """Indice de Gini d'une serie de valeurs positives."""
    x = np.sort(np.asarray(x))
    n = len(x)
    return 2 * np.sum(np.arange(1, n + 1) * x) / (n * np.sum(x)) - (n + 1) / n

print(round(gini(s), 4))

**Réponse 4.** **0,280**.

À titre de repère, l'indice de Gini des revenus disponibles en France tourne autour de 0,29. Notre valeur porte sur des salaires, avant redistribution, et sur un fichier fabriqué : elle n'est comparable à rien de publié.

In [ ]:
print("avec le plafond :", round(gini(s), 4))
print("sans le plafond :", round(gini(s[s < 10000]), 4))
print((s == 10000).sum(), "salaires au plafond")

**Réponse 5.** L'indice passe de **0,280** à **0,263**. Les inégalités n'ont évidemment pas diminué : on vient de **retirer les 72 plus hauts salaires**.

C'est le piège de la question. Un indicateur qui baisse quand on supprime le haut de la distribution ne mesure pas une amélioration : il mesure ce qu'on a retiré.

**Réponse 6.** C'est une **borne basse**.

Les 72 salaires au plafond valent en réalité *plus* de 10 000 euros — on ignore combien. En les ramenant à 10 000, le producteur a comprimé le haut de la distribution, là où se concentre l'inégalité mesurée.

> Tout indicateur d'inégalité calculé sur des données censurées sous-estime l'inégalité réelle. Il faut l'écrire dans la note méthodologique.

## Partie 3 — Croiser deux variables qualitatives

In [ ]:
c = df[df["TEMPS"] != 9]

(100 * pd.crosstab(c["SEXE"], c["TEMPS"], normalize="index")).round(1)

**Réponse 7.** **31,8 %** des femmes travaillent à temps partiel, contre **14,1 %** des hommes — plus du double.

`normalize="index"` calcule les proportions **par ligne**, donc à l'intérieur de chaque sexe. C'est ce qu'on veut : comparer la structure de chaque groupe, et non la répartition de l'ensemble.

In [ ]:
tableau = pd.crosstab(c["SEXE"], c["TEMPS"])
k = stats.chi2_contingency(tableau)
print("khi-deux :", round(k.statistic, 1), "| p-value :", f"{k.pvalue:.1e}")

**Réponse 8.** Khi-deux de **301,5**, p-value de l'ordre de **10⁻⁶⁷**.

**Ce que le test dit :** sexe et temps de travail ne sont pas indépendants.

**Ce qu'il ne dit pas :** ni dans quel sens, ni avec quelle force, ni pourquoi. Sur un échantillon de cette taille, il rejette presque toujours — un écart minuscule suffirait.

Les proportions conditionnelles de la question 7 sont bien plus informatives : elles donnent l'ampleur, 32 % contre 14 %.

## Partie 4 — Relier les deux

In [ ]:
df.groupby("SEXE")["SALAIRE_NET"].mean().round(0)

**Réponse 9.** Hommes **2 781 €**, femmes **2 444 €** : un écart de **336 euros** par mois, soit 12 % du salaire des hommes.

In [ ]:
plein = df[df["TEMPS"] == 1]
print(plein.groupby("SEXE")["SALAIRE_NET"].mean().round(0))

a = plein.loc[plein["SEXE"] == 1, "SALAIRE_NET"].dropna()
b = plein.loc[plein["SEXE"] == 2, "SALAIRE_NET"].dropna()
err = np.sqrt(a.var()/len(a) + b.var()/len(b))
ecart = a.mean() - b.mean()
print("ecart :", round(ecart), "| IC :", (ecart + np.array([-1.96, 1.96]) * err).round(0))
print("p :", f"{stats.ttest_ind(a, b, equal_var=False).pvalue:.1e}")

**Réponse 10.** Parmi les seuls temps complets : hommes **2 974 €**, femmes **2 829 €**. L'écart tombe à **146 euros**, avec un intervalle de confiance de 62 à 229 euros et une p-value de 6 × 10⁻⁴.

**Réponse 11.** L'écart passe de 336 à 146 euros : **plus de la moitié** — 57 % — tient au temps de travail. Les femmes sont deux fois plus souvent à temps partiel, et un temps partiel se paie moins qu'un temps plein.

Peut-on conclure « à travail égal, salaire inégal » sur les 146 euros restants ? **Non, pas encore.** Le temps de travail n'est qu'une des différences entre les deux groupes. Restent au moins le diplôme, le secteur, l'ancienneté, l'âge — que nous n'avons pas tenus fixes.

> Comparer deux groupes en neutralisant une variable à la fois devient vite impraticable. C'est exactement le problème que la **régression** résout, et c'est l'objet des séances à venir.

Ce que l'on peut dire honnêtement : l'écart brut est de 336 euros ; à temps de travail comparable, il reste 146 euros, statistiquement distinct de zéro ; et cette part appelle d'autres explications que nous ne pouvons pas encore départager.

## Pour ceux qui ont terminé

In [ ]:
t = df.groupby("SECTEUR")["SALAIRE_NET"].agg(
        gini=lambda x: gini(x.dropna()), effectif="count").round(3)
t.sort_values("gini", ascending=False)

**Réponse 12.** Le classement est à lire avec les effectifs : un indice de Gini calculé sur deux cents salariés est bien plus instable qu'un indice calculé sur deux mille. C'est le piège de l'effectif de la séance 5, appliqué à un indicateur d'inégalité.

In [ ]:
x = np.sort(s.values)
part_personnes = np.arange(1, len(x) + 1) / len(x)
part_masse     = np.cumsum(x) / x.sum()

import matplotlib.pyplot as plt
plt.plot(part_personnes, part_masse)
plt.plot([0, 1], [0, 1], "--")
plt.xlabel("part cumulée des salariés")
plt.ylabel("part cumulée de la masse salariale")
plt.show()

**Réponse 13.** La diagonale représente l'égalité parfaite. L'aire entre la diagonale et la courbe, rapportée à l'aire sous la diagonale, **est** l'indice de Gini — c'est sa définition géométrique.

**Réponse 14.** À diplôme et temps de travail identiques, l'écart se réduit encore. Mais chaque variable ajoutée découpe l'échantillon en groupes plus petits, donc plus instables : au-delà de deux ou trois, la méthode ne tient plus. D'où la régression.

## Erreurs fréquentes

| Erreur | Symptôme | Ce qu'elle enseigne |
|---|---|---|
| Lire le rapport interdécile comme un rapport de moyennes | « ils gagnent 3,5 fois plus » | Ce sont deux seuils, pas deux moyennes |
| Conclure que l'inégalité baisse sans le plafond | Gini 0,263 < 0,280 | On a retiré le haut de la distribution |
| Oublier `normalize="index"` | proportions sur l'ensemble | On compare la structure de chaque groupe |
| Lire la p-value comme une force de liaison | « très lié, p = 10⁻⁶⁷ » | Le test dit l'existence, pas l'ampleur |
| Conclure « à travail égal » sur 146 € | une seule variable neutralisée | Il en reste beaucoup d'autres |